In [1]:
import sys
sys.path.insert(0, "../src")

import json
import joblib
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

from finnews_sentiment.models.train_model import train_baseline
from finnews_sentiment.features.preprocess import LABEL_NAMES

In [2]:
train_clean = pd.read_csv("data/train_clean.csv")
test_clean = pd.read_csv("data/test_clean.csv")
print(train_clean.shape, test_clean.shape)

(119050, 4) (7599, 4)


In [3]:
# внутри train_baseline: делит train на train/valid (80/20), учит пайплайн, печатает метрики на valid
model, valid_f1 = train_baseline(train_clean)

/Users/mak/Desktop/WORKPLACE/ML_валютная удаленка/Проекты/transformers/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Classification report (validation):
              precision    recall  f1-score   support

           0       0.92      0.90      0.91      5967
           1       0.95      0.97      0.96      5966
           2       0.87      0.87      0.87      5958
           3       0.87      0.89      0.88      5919

    accuracy                           0.91     23810
   macro avg       0.91      0.91      0.91     23810
weighted avg       0.91      0.91      0.91     23810

Macro F1: 0.9057


In [4]:
y_test = test_clean["label"]
y_pred = model.predict(test_clean["text_clean"].astype(str))

acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average="macro")
print("accuracy:", round(acc, 4))
print("macro F1:", round(f1, 4))
print(classification_report(y_test, y_pred, target_names=list(LABEL_NAMES.values())))

accuracy: 0.9021
macro F1: 0.9019
              precision    recall  f1-score   support

       World       0.92      0.90      0.91      1900
      Sports       0.95      0.97      0.96      1900
    Business       0.87      0.87      0.87      1899
    Sci/Tech       0.88      0.88      0.88      1900

    accuracy                           0.90      7599
   macro avg       0.90      0.90      0.90      7599
weighted avg       0.90      0.90      0.90      7599



In [5]:
cm = pd.DataFrame(confusion_matrix(y_test, y_pred),
                  index=list(LABEL_NAMES.values()),
                  columns=list(LABEL_NAMES.values()))
cm

,World,Sports,Business,Sci/Tech
World,1701,61,89,49
Sports,27,1846,13,14
Business,64,20,1644,171
Sci/Tech,58,24,154,1664


In [6]:
import os
os.makedirs("../models", exist_ok=True)
os.makedirs("../results", exist_ok=True)

joblib.dump({"model": model, "meta": {"macro_f1": f1}}, "../models/baseline.joblib")

baseline_results = {
    "model": "tfidf(5000, 1-2gram) + logreg",
    "valid_macro_f1": round(valid_f1, 4),
    "test_accuracy": round(acc, 4),
    "test_macro_f1": round(f1, 4),
}
with open("../results/baseline_metrics.json", "w") as f:
    json.dump(baseline_results, f, indent=2)
baseline_results

{'model': 'tfidf(5000, 1-2gram) + logreg',
 'valid_macro_f1': 0.9057,
 'test_accuracy': 0.9021,
 'test_macro_f1': 0.9019}

In [7]:
bundle = joblib.load("../models/baseline.joblib")
examples = ["oil prices rise after opec meeting", "team win championship final", "new chip faster processor released"]
for t, p in zip(examples, bundle["model"].predict(examples)):
    print(t, "->", LABEL_NAMES[p])

oil prices rise after opec meeting -> Business
team win championship final -> Sports
new chip faster processor released -> Sci/Tech


## Baseline

- модель: TF-IDF (5000 признаков, 1-2 граммы) + Logistic Regression из шаблона
- valid macro F1: ...
- test accuracy: ..., test macro F1: ...
- чаще всего путаются ... и ...
- результаты сохранены в results/baseline_metrics.json, модель в models/baseline.joblib
- дальше сравниваем все улучшения с этими цифрами
